In [35]:
import torch
from torch import nn
from torch.nn import functional as F

triplet_x = torch.randint(0, 2, (10000, 256)).bool()
triplets = torch.randint(0, 256, (5, 3))
y = torch.zeros(10000, dtype=torch.bool)
for i, triplet in enumerate(triplets):
    y = y | (triplet_x[:, triplet[0]] & ~triplet_x[:, triplet[1]] & triplet_x[:, triplet[2]])

In [36]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
import math

# Split data into training and testing sets (80/20 split)
dataset_size = len(triplet_x)
train_size = int(0.8 * dataset_size)
test_size = dataset_size - train_size

# Create dataset and split
dataset = TensorDataset(triplet_x.float(), y.float())
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Input dimension: {triplet_x.shape[1]}")

Training samples: 8000
Test samples: 2000
Input dimension: 256


In [37]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(0), :]

In [38]:
class BinaryTransformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.1):
        super(BinaryTransformer, self).__init__()
        
        self.d_model = d_model
        self.input_dim = input_dim
        
        # Input embedding for binary sequences
        self.input_embedding = nn.Linear(1, d_model)
        # self.positional_encoding = PositionalEncoding(d_model, max_len=input_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(input_dim, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output layers
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        batch_size, seq_len = x.shape
        
        # Reshape to (batch_size, seq_len, 1) for embedding
        x = x.unsqueeze(-1)
        
        # Input embedding
        x = self.input_embedding(x) * math.sqrt(self.d_model)
        x = x + self.positional_encoding.reshape(1, self.input_dim, self.d_model)
        # Add positional encoding
        # x = x.transpose(0, 1)  # (seq_len, batch_size, d_model)
        # x = self.positional_encoding(x)
        # x = x.transpose(0, 1)  # (batch_size, seq_len, d_model)
        
        # Transformer encoding
        x = self.transformer_encoder(x)
        
        # Global average pooling
        x = torch.mean(x, dim=1)  # (batch_size, d_model)
        
        # Classification
        output = self.classifier(x)
        
        return output.squeeze(-1)  # (batch_size,)

In [39]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in tqdm(train_loader, desc="Training"):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            predicted = (outputs > 0.5).float()
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    
    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [40]:
def train_transformer(train_loader, test_loader, input_dim, learning_rate=1e-4, num_epochs=20, 
                     d_model=128, nhead=8, num_layers=4, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"Training on device: {device}")
    
    # Initialize model
    model = BinaryTransformer(
        input_dim=input_dim,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers
    ).to(device)
    
    # Optimizer and loss function
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    criterion = nn.BCELoss()
    
    # Training history for plotting
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
        'epochs': []
    }
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # Training
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Evaluation
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['epochs'].append(epoch + 1)
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    return model, history

In [41]:
def plot_training_history(history):
    """Plot training and validation loss and accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot loss
    ax1.plot(history['epochs'], history['train_loss'], 'b-', label='Training Loss')
    ax1.plot(history['epochs'], history['test_loss'], 'r-', label='Test Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Test Loss')
    ax1.legend()
    ax1.grid(True)
    
    # Plot accuracy
    ax2.plot(history['epochs'], history['train_acc'], 'b-', label='Training Accuracy')
    ax2.plot(history['epochs'], history['test_acc'], 'r-', label='Test Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Test Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

In [42]:
# Train the model
print("Starting training...")
model, history = train_transformer(
    train_loader=train_loader,
    test_loader=test_loader,
    input_dim=triplet_x.shape[1],
    learning_rate=1e-5,
    num_epochs=15,
    d_model=128,
    nhead=8,
    num_layers=4
)

# Plot training history
plot_training_history(history)

Starting training...
Training on device: cuda
Model parameters: 834,433

Epoch 1/15


Training:   0%|          | 0/125 [00:00<?, ?it/s]

Training: 100%|██████████| 125/125 [00:03<00:00, 31.89it/s]


Train Loss: 0.6928, Train Acc: 51.64%
Test Loss: 0.6910, Test Acc: 53.10%

Epoch 2/15


Training: 100%|██████████| 125/125 [00:03<00:00, 32.53it/s]



Train Loss: 0.6921, Train Acc: 52.08%
Test Loss: 0.6915, Test Acc: 53.10%

Epoch 3/15


Training: 100%|██████████| 125/125 [00:03<00:00, 32.35it/s]



Train Loss: 0.6925, Train Acc: 52.49%
Test Loss: 0.6912, Test Acc: 53.10%

Epoch 4/15


Training:   9%|▉         | 11/125 [00:00<00:03, 35.07it/s]



KeyboardInterrupt: 

In [29]:
model.positional_encoding.detach().cpu()[:, 0]

tensor([-1.3384e-04,  7.1565e-04, -8.8885e-05, -3.6537e-03,  5.7424e-04,
         8.2005e-04, -3.3882e-04,  6.9134e-04, -3.4454e-04, -1.2565e-04,
         2.9033e-04, -3.8904e-04,  2.8983e-04,  8.7235e-04,  1.6745e-04,
         3.1607e-03,  5.0753e-04,  3.9295e-04,  6.2067e-05,  5.8396e-06,
         2.8667e-04, -3.6540e-04,  8.1993e-04, -2.4797e-03, -1.6495e-04,
         1.1072e-04,  3.2851e-03, -3.1197e-03,  7.5074e-06, -8.4299e-05,
         6.8530e-05, -8.9907e-04,  1.2468e-03,  2.6484e-04,  9.5294e-05,
         1.0674e-03,  2.8366e-04, -3.7986e-06, -3.0453e-03,  1.9808e-04,
         1.0833e-04,  7.1227e-04,  1.5592e-05,  4.4923e-04,  3.8461e-04,
         2.8578e-04,  2.9197e-04, -7.2318e-05,  6.1656e-04,  5.4677e-04,
         2.4010e-04, -5.1948e-04,  1.9495e-04,  1.4792e-04,  3.0279e-04,
         3.9357e-04,  8.5725e-04, -2.6140e-03, -9.6741e-04,  5.6436e-04,
         2.6490e-04, -3.6623e-04, -1.8883e-05,  2.6727e-03,  6.8278e-04,
         4.0287e-05, -4.6695e-05, -1.1066e-04,  5.6

In [19]:
# Evaluate model performance on a few test samples
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Get a batch of test data
test_batch_x, test_batch_y = next(iter(test_loader))
test_batch_x, test_batch_y = test_batch_x.to(device), test_batch_y.to(device)

with torch.no_grad():
    predictions = model(test_batch_x)
    binary_predictions = (predictions > 0.5).float()

# Show some examples
print("Sample predictions vs actual:")
for i in range(min(10, len(test_batch_x))):
    print(f"Sample {i+1}: Predicted={binary_predictions[i].item():.0f}, Actual={test_batch_y[i].item():.0f}, Confidence={predictions[i].item():.3f}")

# Calculate final accuracy
correct = (binary_predictions == test_batch_y).sum().item()
total = len(test_batch_y)
print(f"\nBatch accuracy: {100 * correct / total:.2f}%")

Sample predictions vs actual:
Sample 1: Predicted=1, Actual=0, Confidence=0.728
Sample 2: Predicted=1, Actual=1, Confidence=0.728
Sample 3: Predicted=1, Actual=1, Confidence=0.729
Sample 4: Predicted=1, Actual=0, Confidence=0.729
Sample 5: Predicted=1, Actual=0, Confidence=0.730
Sample 6: Predicted=1, Actual=1, Confidence=0.728
Sample 7: Predicted=1, Actual=1, Confidence=0.727
Sample 8: Predicted=1, Actual=0, Confidence=0.730
Sample 9: Predicted=1, Actual=1, Confidence=0.726
Sample 10: Predicted=1, Actual=1, Confidence=0.729

Batch accuracy: 68.75%
